In [ ]:
# Imports
import json
import tifffile as tiff
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from tqdm import tqdm
import os

In [ ]:
# Paths
gt_path = r'C:\Users\kiril\Desktop\OPCal\Tal&Meital code\20250409_1_ATP_100uM\20250409_1_ATP_100uM_ROI_All.tif'  # Update path
cnmf_summary_mask_path = r'C:\Users\kiril\Desktop\OPCal\Tal&Meital code\Run001_20250409_1_ATP_100uM_diameter20um\20250409_1_ATP_100uM_TIF_VIDEO_coco_summary_binary_mask.png'
coco_json_path = r'C:\Users\kiril\Desktop\OPCal\Tal&Meital code\Run001_20250409_1_ATP_100uM_diameter20um\20250409_1_ATP_100uM_TIF_VIDEO_coco.json'
movie=tiff.imread(r'C:\Users\kiril\Desktop\OPCal\Tal&Meital code\20250409_1_ATP_100uM_TIF_VIDEO.tif')

In [ ]:
# Load CNMF summary mask and compute global IoU
cnmf_mask = cv2.imread(cnmf_summary_mask_path, 0) > 0
# Load ground truth mask
gt = cv2.imread(gt_path, 0)
gt_bin = gt.copy()
gt_mask =  gt_bin > 0

intersection = np.logical_and(gt_mask, cnmf_mask).sum()
union = np.logical_or(gt_mask, cnmf_mask).sum()
global_iou = intersection / union if union else 0

print(f"🌍 Global IoU (full mask): {global_iou:.4f}")

In [ ]:
# Load coco.json for object-level comparison?
with open(coco_json_path, 'r') as f:
    coco = json.load(f)

image_info = coco['images'][0]
height, width = image_info['height'], image_info['width']
annotations = coco['annotations']

# Reconstruct CNMF masks from polygons
cnmf_masks = []
cnmf_centers = []

for ann in annotations:
    poly_pts = np.array(ann['segmentation'][0]).reshape(-1, 2)
    poly_pts = np.round(poly_pts).astype(np.int32)
    
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(mask, [poly_pts], 1)
    cnmf_masks.append(mask > 0)
    
    # Center from moments
    M = cv2.moments(mask)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        cnmf_centers.append((cx, cy))

# Extract GT contours and centers
gt_contours, _ = cv2.findContours(gt_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
gt_masks = []
gt_centers = []

for c in gt_contours:
    area = cv2.contourArea(c)
    if area < 10:
        continue
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.drawContours(mask, [c], -1, 1, -1)
    gt_masks.append(mask > 0)

    M = cv2.moments(c)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        gt_centers.append((cx, cy))

In [ ]:
# Match each GT center to the closest CNMF center within a threshold.
# Greedy matching ensures one-to-one matching.

gt_centers = np.array(gt_centers)
cnmf_centers = np.array(cnmf_centers)

distances = cdist(gt_centers, cnmf_centers)
um_per_pixel = 636.416 / 512  # 1.243
diameter_um = 5 # 10 microns
diameter_px = diameter_um / um_per_pixel  # ~8.04
threshold = 1.5 * diameter_px  # ≈ 12 pixels

gt_matched = np.zeros(len(gt_centers), dtype=bool)
cnmf_matched = np.zeros(len(cnmf_centers), dtype=bool)
matches = []

while True:
    mask = (~gt_matched[:, None]) & (~cnmf_matched[None, :]) & (distances < threshold)
    if not np.any(mask):
        break
    i, j = np.unravel_index(np.argmin(distances * mask + (~mask) * 1e6), distances.shape)
    gt_matched[i] = True
    cnmf_matched[j] = True
    matches.append((i, j))

print(np.shape(gt_centers))


In [ ]:
# Object-level metrics
TP = len(matches) # True Positives: matched components
FP = np.sum(~cnmf_matched) # False Positives: CNMF cells not matched
FN = np.sum(~gt_matched) # False Negatives: GT cells not matched

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

# Calculate mean Intersection-over-Union (IoU) for matched masks
ious = []
for i, j in matches:
    inter = np.logical_and(gt_masks[i], cnmf_masks[j]).sum()
    union = np.logical_or(gt_masks[i], cnmf_masks[j]).sum()
    ious.append(inter / union if union else 0)

mean_iou = np.mean(ious) if ious else 0

print("\n📊 Object-Level Evaluation:")
print(f"True Positives: {TP}")
print(f"False Positives: {FP}")
print(f"False Negatives: {FN}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Mean IoU: {mean_iou:.4f}")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import cv2

#background image
mean_projection = np.mean(movie, axis=0)
vmin = np.percentile(mean_projection, 50)
vmax = np.percentile(mean_projection, 99.5)
mean_proj_norm = np.clip((mean_projection - vmin) / (vmax - vmin), 0, 1)
background_image=mean_proj_norm

plt.figure(figsize=(12, 12))

# 1. Show background (projection image)
plt.imshow(background_image, cmap='gray')

# 2. Ground Truth contours (green)
for mask in gt_masks:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        if c.shape[0] > 2:
            c = c.squeeze()
            plt.plot(c[:, 0], c[:, 1], color='lime', linewidth=1.2)

# 3. CNMF contours (red)
for mask in cnmf_masks:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        if c.shape[0] > 2:
            c = c.squeeze()
            plt.plot(c[:, 0], c[:, 1], color='red', linewidth=1.2)

# 4. Matching lines (blue dashed)
for i, j in matches:
    gt_pt = gt_centers[i]
    cnmf_pt = cnmf_centers[j]
    plt.plot([gt_pt[0], cnmf_pt[0]], [gt_pt[1], cnmf_pt[1]], 'b--', linewidth=0.8)

# 5. Centers
for x, y in gt_centers:
    plt.plot(x, y, 'go', markersize=4, label='GT Centers' if 'GT Centers' not in plt.gca().get_legend_handles_labels()[1] else "")

for x, y in cnmf_centers:
    plt.plot(x, y, 'r+', markersize=4, label='CNMF Centers' if 'CNMF Centers' not in plt.gca().get_legend_handles_labels()[1] else "")

# 6. Title and legend
plt.title("Evaluation: Ground Truth (green) vs CNMF (red) with matches (blue lines)", fontsize=14)
plt.axis('off')

legend_elements = [
    Line2D([0], [0], color='lime', lw=1.5, label='Ground Truth Contours'),
    Line2D([0], [0], color='red', lw=1.5, label='CNMF Contours'),
    Line2D([0], [0], color='blue', lw=1, linestyle='--', label='Matches'),
    Line2D([0], [0], marker='o', color='g', label='GT Centers', markersize=6, linestyle='None'),
    Line2D([0], [0], marker='+', color='r', label='CNMF Centers', markersize=6, linestyle='None')
]

plt.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.25, 1))
plt.tight_layout()
plt.show()


In [ ]:
# plot bar chart ---
labels = ['Global IoU', 'Pixel Recall (GT)', 'Pixel Precision (CNMF)']
values = [global_iou, recall, precision]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values, color=['steelblue', 'orange', 'seagreen'])
plt.ylim(0, 1)
plt.ylabel('Score')
plt.title('Pixel-Level Evaluation Metrics')

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., height + 0.02,
             f'{height:.2f}', ha='center', fontsize=12)

plt.tight_layout()
plt.show()